# Dunnhumby preference-preserving joint M2 v1.5

Dunnhumby seed 42 validation에서 외부 M1@64와 `joint_nv_preference_preserving`만 비교합니다. ID는 순수 ID-BPR로, N/V는 `stopgrad(S_ID)`를 조건으로 한 별도 BPR로 한 모델·한 optimizer·한 batch loop에서 동시 학습합니다. test와 holdout은 열지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REVIEWED_SHA = '0bfc552d1a70612f6a391baf2401aaeb8cde7920'
repo = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run([
    'git', 'clone', '-q',
    'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)
], check=True)
os.chdir(repo)
subprocess.run(['git', 'checkout', '-q', REVIEWED_SHA], check=True)
assert subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip() == REVIEWED_SHA
print('code:', REVIEWED_SHA)


In [ ]:
import importlib, json, sys, torch
for module_name, module in list(sys.modules.items()):
    module_path = getattr(module, '__file__', '') or ''
    if module_path and str(repo) in str(module_path):
        del sys.modules[module_name]
importlib.invalidate_caches()

from IPython.display import display
import pandas as pd
from lightgcn_clv_joint_nv import (
    configure_preference_preserving_dunnhumby_run,
    preflight_summary,
    run_experiment,
)
assert torch.cuda.is_available(), (
    '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
cfg = configure_preference_preserving_dunnhumby_run()
summary = preflight_summary(cfg)
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['code_version'] == 'm2-joint-nv-lightgcn-v1.5'
assert summary['dataset'] == 'dunnhumby'
assert summary['models'] == ['m1', 'joint_nv_preference_preserving']
assert summary['loss']['type'] == 'preference_preserving_joint_bpr'
assert summary['loss']['sample_weighting'] is False
assert summary['gamma']['initial_score_strength'] == 0.1
assert summary['eval_test'] is False
assert summary['eval_holdout'] is False
assert cfg.out_dir.endswith('_m2_joint_nv_preference_preserving_v15')
print('설정 확인 완료. 다음 셀은 M1과 수정 M2 학습을 시작합니다.')


In [ ]:
result_df = run_experiment(cfg)


In [ ]:
columns = [
    'model_id', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'revenue@20', 'revenue@50',
    'arp@10', 'coverage@10', 'n_distinct@10',
    'exposure_entropy@10', 'eff_catalog@10',
    'top10_share@10', 'top100_share@10', 'value_alignment',
    'gamma_n', 'gamma_v',
]
absolute = result_df[[c for c in columns if c in result_df.columns]].copy()
print('===== 절대지표 =====')
display(absolute)

indexed = result_df.set_index('model_id')
m1 = indexed.loc['m1']
m2 = indexed.loc['joint_nv_preference_preserving']
metric_names = [
    c for c in columns[1:]
    if c in result_df.columns and c not in {'gamma_n', 'gamma_v'}
]
comparison = pd.DataFrame({
    'metric': metric_names,
    'M1': [m1[c] for c in metric_names],
    'preference_preserving_M2_v15': [m2[c] for c in metric_names],
})
comparison['absolute_delta'] = (
    comparison['preference_preserving_M2_v15'] - comparison['M1']
)
comparison['relative_change_pct'] = (
    comparison['absolute_delta']
    / comparison['M1'].replace(0, float('nan')) * 100
)
print('===== M1 대비 변화 =====')
display(comparison)
print('최종 gamma:', {'N': m2.get('gamma_n'), 'V': m2.get('gamma_v')})
print('판정:', result_df.attrs['decision'])
print('결과 파일:', result_df.attrs['result_paths'])
print('완료. 위 절대지표·변화표·최종 gamma를 공유해 주세요.')
